# ⚠️ RAG Experiment Result — Why Answer Generation Failed

## 📌 Objective

The goal of this experiment was to build a Retrieval-Augmented Generation (RAG) system using a synthetic dataset of customer queries.

We successfully implemented:

* document creation from CSV
* embeddings using HuggingFace
* FAISS vector store
* semantic retrieval
* LLM-based answer generation

---

## ✅ What Worked

### 1. Semantic Retrieval

The system correctly retrieves relevant documents based on query meaning.

Example:

* Query: *"Can I cancel my order?"*
* Retrieved results:

  * "Can you help me cancel my order?"
  * "Can I cancel my order before it ships?"

All results correctly map to the label:
👉 `cancel_request`

### 2. Embedding Quality

Similar intents are clustered correctly in vector space:

* "refund" queries group together
* "order status" queries group together

### 3. FAISS Vector Search

Efficient similarity search is working as expected.

---

## ❌ What Did NOT Work

### Answer Generation (RAG QA)

When using an LLM to generate answers from retrieved context, the system failed to produce grounded responses.

### Observed Behavior

The model generated answers like:

* "Contact customer support"
* "Visit the website"
* "Call the service number"

👉 These were **not present in the retrieved context**

---

## 🧠 Root Cause

### 1. Dataset Type Mismatch

The dataset contains:

* user queries / intents

Example:

* "I need a refund"
* "Can I cancel my order?"

But RAG QA requires:

* factual or instructional knowledge

Example:

* "Refunds can be requested within 30 days"
* "Orders can be canceled before shipping"

---

### 2. Missing Answerable Content

The retrieved documents do NOT contain:

* procedures
* policies
* instructions
* factual answers

So the LLM has no valid information to generate from.

---

### 3. LLM Behavior

Even with strict prompts:

* the model attempts to be helpful
* fills missing information using prior knowledge

This leads to:
👉 hallucination (ungrounded answers)

---

## 🎯 Key Insight

> RAG performance depends more on **data quality** than on embeddings, vector DB, or model choice.

---

## 📊 System Classification

This system is better described as:

👉 **Semantic Retrieval / Intent Matching System**

Not:

❌ Full QA RAG System

---

## 🔄 Correct Use Case for This Dataset

This dataset is well-suited for:

* intent classification via retrieval
* semantic search
* nearest-neighbor matching
* label prediction

---

## 🚀 Next Steps

To enable true RAG QA:

We need a new dataset with:

* structured knowledge
* policies
* instructions
* FAQs

Example:

```
# Refund Policy
Customers can request a refund within 30 days...

# Cancellation Policy
Orders can be canceled before shipping...
```

This will be implemented in:

👉 `03_rag_unstructured_docs.ipynb`

---

## 🧠 Final Takeaway

> RAG is not just about retrieval — it requires **answerable knowledge in the corpus**.

Without that, even perfect retrieval will not produce meaningful answers.


In [1]:
import os
import pandas as pd
import numpy as np
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [2]:
DATA_DIR = "/mnt/data/llm_project/llm-text-learning-lab/data/synthetic"
csv_path = os.path.join(DATA_DIR, "synthetic_clean_300.csv")

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (260, 3)
Columns: ['text', 'label', 'source']


,text,label,source
0,I need to check the status of my latest order.,order_status,synthetic
1,Can I cancel my order that I placed yesterday?,cancel_request,synthetic
2,I received the wrong item and I want a refund.,refund_request,synthetic
3,Is there a discount available for the product ...,product_question,synthetic
4,I am dissatisfied with the service I received.,complaint,synthetic


In [3]:
documents = []

for idx, row in df.iterrows():
    doc = Document(page_content=row["text"],metadata={"label": row["label"], "source": row["source"], "row_id": idx})
    documents.append(doc)

print("Total documents:", len(documents))
print("\nSample document:\n")
print(documents[20])

Total documents: 260

Sample document:

page_content='I need to check the status of my order number 12345.' metadata={'label': 'order_status', 'source': 'synthetic', 'row_id': 20}


In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embedding_model)

print("FAISS vector store created successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store created successfully


In [5]:
sample_text = documents[0].page_content

vector = embedding_model.embed_query(sample_text)

print("Text:\n", sample_text)
print("\nVector length:", len(vector))
print("\nFirst 30 values of vector:\n", vector[:30])

Text:
 I need to check the status of my latest order.

Vector length: 384

First 30 values of vector:
 [-0.006082481704652309, -0.06009793281555176, 0.021092534065246582, -0.04928981512784958, -0.01158025674521923, -0.03573612496256828, -0.09638022631406784, -0.04045303165912628, -0.04018951579928398, 0.0545257069170475, 0.05641309916973114, 0.09317878633737564, -0.05445154383778572, -0.08073035627603531, -0.05348467454314232, -0.061746276915073395, 0.050371699035167694, -0.009908054955303669, -0.05302077904343605, -0.03217679262161255, -0.03729983791708946, 0.038040097802877426, -0.06373611837625504, 0.024243338033556938, -0.03590289130806923, 0.02269843965768814, -0.1016196757555008, -0.012198282405734062, -0.017179330810904503, -0.08688513934612274]


In [6]:
text1 = "I need to check the status of my latest order."
text2 = "Where is my package right now?"
text3 = "I want a refund for my purchase."

v1 = embedding_model.embed_query(text1)
v2 = embedding_model.embed_query(text2)
v3 = embedding_model.embed_query(text3)

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("text1 vs text2:", cosine_similarity(v1, v2))
print("text1 vs text3:", cosine_similarity(v1, v3))

text1 vs text2: 0.3428154825254759
text1 vs text3: 0.2951711234911559


In [7]:
query = "Where is my order?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Text:", doc.page_content)
    print("Label:", doc.metadata["label"])


--- Result 1 ---
Text: I can't find my order number, can you help?
Label: order_status

--- Result 2 ---
Text: What is the status of my order?
Label: order_status

--- Result 3 ---
Text: What’s the status of my order?
Label: order_status


In [8]:
test_queries = [
    "I want a refund for my order",
    "Can I cancel the purchase?",
    "Your service was very disappointing",
    "Do you have any discount offers?"
]

for query in test_queries:
    print(f"\n\nQUERY: {query}")
    results = vectorstore.similarity_search(query, k=3)

    for i, doc in enumerate(results, 1):
        print(f"\n--- Result {i} ---")
        print("Text:", doc.page_content)
        print("Label:", doc.metadata["label"])



QUERY: I want a refund for my order

--- Result 1 ---
Text: How can I request a refund for my order?
Label: refund_request

--- Result 2 ---
Text: I need a refund for my purchase
Label: refund_request

--- Result 3 ---
Text: I want to cancel my order and get a refund for it.
Label: cancel_request


QUERY: Can I cancel the purchase?

--- Result 1 ---
Text: I want to cancel my purchase because I changed my mind
Label: cancel_request

--- Result 2 ---
Text: Can I cancel my order before it ships?
Label: cancel_request

--- Result 3 ---
Text: I want to cancel my order and return the product.
Label: cancel_request


QUERY: Your service was very disappointing

--- Result 1 ---
Text: I am dissatisfied with the service I received.
Label: complaint

--- Result 2 ---
Text: I received my order late and I'm disappointed with the service.
Label: complaint

--- Result 3 ---
Text: I am very disappointed with the late delivery and poor service.
Label: complaint


QUERY: Do you have any discount offer

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "I want a refund for my order"
retrieved_docs = retriever.invoke(query)

print("Query:", query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Doc {i} ---")
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)

Query: I want a refund for my order

--- Retrieved Doc 1 ---
Text: How can I request a refund for my order?
Metadata: {'label': 'refund_request', 'source': 'synthetic', 'row_id': 231}

--- Retrieved Doc 2 ---
Text: I need a refund for my purchase
Metadata: {'label': 'refund_request', 'source': 'synthetic', 'row_id': 98}

--- Retrieved Doc 3 ---
Text: I want to cancel my order and get a refund for it.
Metadata: {'label': 'cancel_request', 'source': 'synthetic', 'row_id': 121}


In [10]:
def format_context(docs):
    return "\n\n".join([doc.page_content for doc in docs])

query = "I want a refund for my order"
retrieved_docs = retriever.invoke(query)
context = format_context(retrieved_docs)

print("QUERY:")
print(query)

print("\nRETRIEVED CONTEXT:")
print(context)

QUERY:
I want a refund for my order

RETRIEVED CONTEXT:
How can I request a refund for my order?

I need a refund for my purchase

I want to cancel my order and get a refund for it.


In [11]:

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print("GROQ API KEY loaded:", groq_api_key[:6], "..." if groq_api_key else "❌ Not found")

GROQ API KEY loaded: gsk_Jv ...


In [12]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [13]:
query = "Can I cancel my order?"

retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
You are a helpful and friendly customer support assistant.

Rules:
- Use ONLY the provided context to answer the question.
- Do NOT use any external knowledge.
- Do NOT make assumptions.
- If the answer is not explicitly in the context, say:
  "I could not find that in the knowledge base."

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print("ANSWER:\n")
print(response.content)

print("\nRETRIEVED SOURCES:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Source {i}:")
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)
    print()

ANSWER:

You've already asked me to cancel your order. I could not find that in the knowledge base.

RETRIEVED SOURCES:

Source 1:
Text: Can you help me cancel my order?
Metadata: {'label': 'cancel_request', 'source': 'synthetic', 'row_id': 78}

Source 2:
Text: Can I cancel my order before it ships?
Metadata: {'label': 'cancel_request', 'source': 'synthetic', 'row_id': 106}

Source 3:
Text: I want to cancel my order
Metadata: {'label': 'cancel_request', 'source': 'synthetic', 'row_id': 97}



In [14]:
query = "Can I cancel my order?"

retrieved_docs = retriever.invoke(query)

top_doc = retrieved_docs[0]

print("QUERY:")
print(query)

print("\nTOP MATCH:")
print(top_doc.page_content)

print("\nPREDICTED LABEL:")
print(top_doc.metadata["label"])

print("\nALL RETRIEVED RESULTS:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Result {i} ---")
    print("Text:", doc.page_content)
    print("Label:", doc.metadata["label"])

QUERY:
Can I cancel my order?

TOP MATCH:
Can you help me cancel my order?

PREDICTED LABEL:
cancel_request

ALL RETRIEVED RESULTS:

--- Result 1 ---
Text: Can you help me cancel my order?
Label: cancel_request

--- Result 2 ---
Text: Can I cancel my order before it ships?
Label: cancel_request

--- Result 3 ---
Text: I want to cancel my order
Label: cancel_request
